# 🏆 Churn Prediction: Master Production Pipeline (LIVE & FULL AUDIT)
**Core Priority**: RECALL Optimization.
**Transparency**: MLflow Autologging & Verbose Search enabled.

In [ ]:
# 1. Setup & Environment
!nvidia-smi
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub -q

import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU: print(f"✅ CUDA: {torch.cuda.get_device_name(0)}")

import opendatasets as od
import os, json, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, f1_score, confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from mlflow.models import infer_signature

# ACTIVATE LIVE AUTOLOGGING
mlflow.sklearn.autolog(log_models=False)

In [ ]:
# 2. Auth & Data Loading
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

def clean_dataset(df):
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid', 'last_interaction'] if c in df.columns])
    if 'age' in df.columns: df = df[(df['age'] >= 18) & (df['age'] <= 120)]
    for col in [c for c in ['total_spend', 'totalcharges'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['churn'])

if not os.path.exists("customer-churn-dataset"):
    od.download("https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset")

df_raw = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-training-master.csv"))
df_te = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-testing-master.csv"))
df_tr, df_va = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=df_raw['churn'])
print(f"✅ Data Loaded: {len(df_tr)} Train | {len(df_te)} Test")

## 3. Live Training Cycle & Audit

In [ ]:
def run_live_experiment(df_tr, df_va, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), X_tr.select_dtypes(include=[np.number]).columns.tolist()),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), X_tr.select_dtypes(include=['object']).columns.tolist())
    ])
    
    if model_type == "xgboost":
        pw = (y_tr == 0).sum() / (y_tr == 1).sum()
        clf = XGBClassifier(scale_pos_weight=pw, device='cuda' if HAS_GPU else 'cpu', tree_method='hist')
        params = {'clf__max_depth': [3, 5], 'clf__n_estimators': [100, 200]}
    elif model_type == "random_forest":
        clf = RandomForestClassifier(random_state=42)
        params = {'clf__max_depth': [10, 20], 'clf__n_estimators': [100]}
    else:
        clf = LogisticRegression(class_weight='balanced', max_iter=2000)
        params = {'clf__C': [0.1, 1.0]}
        
    mlflow.set_experiment("Churn_Live_Audit_V2")
    with mlflow.start_run(run_name=f"LIVE_{model_type.upper()}"):
        print(f"\n🔥 STARTING EXPERIMENT: {model_type.upper()}")
        pipeline = Pipeline([('pre', preprocessor), ('clf', clf)])
        
        # VERBOSE=2 for live progress updates
        search = RandomizedSearchCV(pipeline, params, n_iter=2, cv=2, scoring='recall', verbose=2)
        search.fit(X_tr, y_tr)
        
        best_model = search.best_estimator_
        print(f"✅ Best Params found: {search.best_params_}")
        
        y_pred = best_model.predict(X_te)
        y_probs = best_model.predict_proba(X_te)[:,1] if hasattr(best_model, 'predict_proba') else y_pred
        
        # Audit Visuals
        fpr, tpr, _ = roc_curve(y_te, y_probs)
        plt.figure(figsize=(6,4)); plt.plot(fpr, tpr, label=f'AUC={auc(fpr,tpr):.2f}'); plt.title(f"ROC: {model_type}"); plt.legend(); plt.show()
        
        # Fairness & SHAP
        if 'gender' in df_te.columns:
            df_audit = df_te.copy(); df_audit['err'] = np.abs(y_te - y_pred)
            gap = df_audit.groupby('gender')['err'].mean().diff().abs().iloc[-1]
            mlflow.log_metric("gender_bias_gap", gap)
            print(f"⚖️ Gender Bias Gap: {gap:.4f}")
            
        try:
            print("🧠 Generating SHAP insights...")
            sample = X_te.sample(min(100, len(X_te)))
            feature_names = best_model.named_steps['pre'].get_feature_names_out()
            X_mapped = pd.DataFrame(best_model.named_steps['pre'].transform(sample), columns=feature_names)
            explainer = shap.Explainer(best_model.named_steps['clf'])
            shap_values = explainer(X_mapped)
            plt.figure(figsize=(10,6)); shap.summary_plot(shap_values, X_mapped, show=False)
            plt.savefig(f"shap_{model_type}.png"); mlflow.log_artifact(f"shap_{model_type}.png"); plt.show()
        except: pass

        signature = infer_signature(X_tr, best_model.predict(X_tr))
        mlflow.sklearn.log_model(best_model, "model", registered_model_name=f"CustomerChurnModel_{model_type}", signature=signature)
        print(f"🏆 {model_type.upper()} COMPLETE. Registered URI: models:/CustomerChurnModel_{model_type}/latest")

for m in ["xgboost", "logistic_regression", "random_forest"]:
    run_live_experiment(df_tr, df_va, df_te, m)